In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine

# Ensure processed directory exists
os.makedirs('../data/processed', exist_ok=True)
engine = create_engine('sqlite:///../data/processed/bluestock_mf.db')

raw_files = [
    "01_fund_master.csv", "02_nav_history.csv", "03_aum_by_fund_house.csv",
    "04_monthly_sip_inflows.csv", "05_category_inflows.csv", "06_industry_folio_count.csv",
    "07_scheme_performance.csv", "08_investor_transactions.csv", "09_portfolio_holdings.csv",
    "10_benchmark_indices.csv"
]

print("Cleaning and exporting all 10 datasets to data/processed/ ...")

for file in raw_files:
    raw_path = f"../data/raw/{file}"
    # Name the output file
    processed_path = f"../data/processed/cleaned_{file}"
    
    # Load raw data
    try:
        df = pd.read_csv(raw_path)
    except FileNotFoundError:
        print(f"File missing, skipping: {raw_path}")
        continue

    # 1. Specialized cleaning: Fund Master
    if "01_fund_master" in file:
        df = df.rename(columns={'scheme_name': 'fund_name'})
        df['launch_date'] = pd.to_datetime(df['launch_date'], errors='coerce')
        # Insert to DB
        df[['amfi_code', 'fund_name', 'fund_house', 'category', 'launch_date']].to_sql(
            'dim_fund', con=engine, if_exists='replace', index=False
        )

    # 2. Specialized cleaning: NAV History
    elif "02_nav_history" in file:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df = df.drop_duplicates().dropna(subset=['nav'])
        df = df[df['nav'] > 0].sort_values(['amfi_code', 'date'])
        # Insert to DB
        df.to_sql('fact_nav', con=engine, if_exists='replace', index=False)

    # 3. Specialized cleaning: Scheme Performance
    elif "07_scheme_performance" in file:
        df = df.rename(columns={
            'return_1yr_pct': 'return_1y', 
            'return_3yr_pct': 'return_3y', 
            'return_5yr_pct': 'return_5y', 
            'expense_ratio_pct': 'expense_ratio'
        })
        # Flag anomalies
        df['anomaly_flag'] = df[['return_1y', 'return_3y', 'return_5y']].apply(
            lambda x: (x > 100) | (x < -90)
        ).any(axis=1).astype(int)
        # Insert to DB
        df[['amfi_code', 'return_1y', 'return_3y', 'return_5y', 'expense_ratio', 'aum_crore', 'anomaly_flag']].to_sql(
            'fact_performance', con=engine, if_exists='replace', index=False
        )

    # 4. Specialized cleaning: Investor Transactions
    elif "08_investor_transactions" in file:
        df = df.rename(columns={'transaction_date': 'date', 'amount_inr': 'amount'})
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df['transaction_type'] = df['transaction_type'].str.strip().str.upper()
        df = df[df['amount'] > 0]
        df['kyc_status'] = df['kyc_status'].str.strip().str.upper()
        # Insert to DB
        df[['amfi_code', 'investor_id', 'date', 'transaction_type', 'amount', 'state', 'kyc_status']].to_sql(
            'fact_transactions', con=engine, if_exists='replace', index=False
        )

    # --- THE FIX: EXPORT EVERY FILE TO CSV ---
    # This runs for all 10 files, ensuring your deliverables are generated
    df.to_csv(processed_path, index=False)
    print(f"Saved: {processed_path}")

print("\nSuccess! Check your 'data/processed' folder for all 10 cleaned CSV files and the SQLite DB.")

Cleaning and exporting all 10 datasets to data/processed/ ...
Saved: ../data/processed/cleaned_01_fund_master.csv
Saved: ../data/processed/cleaned_02_nav_history.csv
Saved: ../data/processed/cleaned_03_aum_by_fund_house.csv
Saved: ../data/processed/cleaned_04_monthly_sip_inflows.csv
Saved: ../data/processed/cleaned_05_category_inflows.csv
Saved: ../data/processed/cleaned_06_industry_folio_count.csv
Saved: ../data/processed/cleaned_07_scheme_performance.csv
Saved: ../data/processed/cleaned_08_investor_transactions.csv
Saved: ../data/processed/cleaned_09_portfolio_holdings.csv
Saved: ../data/processed/cleaned_10_benchmark_indices.csv

Success! Check your 'data/processed' folder for all 10 cleaned CSV files and the SQLite DB.
